# SCM-VLM: train on Google Colab

Before you start: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Checkpoints, the feature cache and results are stored in your Google Drive (`MyDrive/scm_vlm/`),
so nothing is lost if Colab disconnects. The dataset (~13 GB) is downloaded fresh each session
(about 5-10 minutes).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

## 2. Mount Google Drive (keeps checkpoints between sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/scm_vlm'
import os
for sub in ['checkpoints', 'cache', 'results']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)

## 3. Get the code

Push your latest code to GitHub first (from your laptop: `git add -A`, `git commit`, `git push`).
If the repository is **private**, replace the URL with
`https://<your-github-token>@github.com/MANDEEP-SAI-SURYA/Intervention-Aware-Multimodal-Reasoning-using-SCM.git`
(create a token at GitHub → Settings → Developer settings → Personal access tokens).

In [ ]:
REPO = 'https://github.com/MANDEEP-SAI-SURYA/Intervention-Aware-Multimodal-Reasoning-using-SCM.git'
%cd /content
!rm -rf scm && git clone -q $REPO scm
%cd /content/scm
!git log --oneline | head -3

## 4. Install dependencies (PyTorch is already installed on Colab)

In [ ]:
!pip install -q transformers timm networkx seaborn openai

## 5. Keep checkpoints / cache / results on Drive

In [ ]:
%cd /content/scm
!rm -rf checkpoints cache results
!ln -s $DRIVE_DIR/checkpoints checkpoints
!ln -s $DRIVE_DIR/cache cache
!ln -s $DRIVE_DIR/results results
!ls -l | grep -- '->'

## 6. Download VQA v2 + COCO train2014 (~13 GB, 5-10 min)

The COCO object annotations (`instances_train2014.json`) are also downloaded; they are needed
for the counterfactual-supervision step later.

If a download fails (404/403), get the current links from https://visualqa.org/download.html (*Training questions* and *Training annotations*, VQA v2) and https://cocodataset.org/#download (*2014 Train images* and *2014 Train/Val annotations*), and paste them into the cell below.

In [ ]:
%cd /content/scm
!mkdir -p coco/vqa coco/train2014 coco/annotations
# VQA v2 questions + answers
!wget -q --show-progress https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip -O /content/q.zip
!wget -q --show-progress https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip -O /content/a.zip
!unzip -q -o /content/q.zip -d coco/vqa && unzip -q -o /content/a.zip -d coco/vqa
# COCO object annotations
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2014.zip -O /content/ann.zip
!unzip -q -o /content/ann.zip -d coco
# COCO train2014 images  ->  coco/train2014/train2014/*.jpg
!wget -q --show-progress http://images.cocodataset.org/zips/train2014.zip -O /content/train2014.zip
!unzip -q -o /content/train2014.zip -d coco/train2014 && rm /content/train2014.zip
!ls coco/vqa coco/annotations | head; ls coco/train2014/train2014 | wc -l

## 7. Check that questions and images match

In [ ]:
!python validate_dataset.py

> **Retrain after the DETR feature fix.** Earlier versions took each object's embedding from the
> wrong DETR query, so node features did not match the detected objects. `train.py` now recomputes
> the cached features automatically on the first epoch (old cache files are ignored), and both the
> full model (step 8) and the no-graph model (step 8b) must be retrained. Old results are not comparable.

## 8. Train

Short run first (step 1 of the plan). The first epoch runs DETR on every image and fills the cache
on Drive; later epochs (and later sessions) reuse it and are much faster.

For the full run later: `!python train.py --samples 20000 --epochs 10`

In [ ]:
!python train.py --samples 3000 --epochs 5

In [ ]:
import pandas as pd
display(pd.read_csv('results/training_history.csv'))
display(pd.read_csv('results/final_metrics.csv'))

## 8b. Ablation: remove the causal graph

Trains the same model **without the SCM graph**: no edge scorer, no adjacency matrix, no message passing
between objects, and no DAG / sparsity / edge losses. Everything else (DETR, BERT, question-guided answer
head, counterfactual loss, data split) stays the same, so any difference comes from the graph.

Use the **same `--samples` and `--epochs`** as the full run in step 8 so both models see the same
train/validation questions. Results go to separate files (`*_nograph.csv`, `scm_vlm_latest_nograph.pt`),
so the full model's results are not overwritten. The DETR cache from step 8 is reused, so this run is faster.

> The full model must have been trained with this version of `train.py` (it now also writes
> `results/val_predictions.csv`). If you trained it earlier, re-run step 8 first.

In [ ]:
!python train.py --samples 3000 --epochs 5 --graph-mode none

In [ ]:
import pandas as pd
full = pd.read_csv('results/final_metrics.csv'); full.insert(0, 'Model', 'Full SCM-VLM')
nog = pd.read_csv('results/final_metrics_nograph.csv'); nog.insert(0, 'Model', 'Without causal graph')
display(pd.concat([full, nog], ignore_index=True))

### How do the answers differ?
Accuracy per answer type (yes/no, number, other), agreement between the two models, questions only one of
them gets right (with a McNemar significance test) and example questions where the answers differ.

In [ ]:
!python compare_answers.py --examples 30

In [ ]:
!python compare_answers.py --image images/Man_riding_bicycle.png \
    --question "What is the man riding?" \
    --question "What is the man holding?" \
    --question "How many cars are there?" \
    --question "Is the traffic light red?"

## 8c. Does the causal graph learn relationships? (paper evidence)

Run steps 8 and 8b first (after the feature fix). This section

1. trains two more seeds of each model, so a gap can be told apart from run-to-run noise,
2. compares accuracy on **relational questions** (spatial: *next to, on, behind ...*; interaction:
   *riding, holding, wearing ...*) vs. non-relational ones, with a significance test per group,
3. runs `relation_analysis.py`: graph knock-out on the trained full model, what the learned edges
   connect, and linear probes that test whether an object's representation knows about its related
   objects,
4. shows **how each model reasons** on single images (`explain_reasoning.py`).

In [ ]:
# 2 extra seeds per model (same data split); ~2x the time of steps 8 + 8b
for seed in [1, 2]:
    !python train.py --samples 3000 --epochs 5 --seed {seed}
    !python train.py --samples 3000 --epochs 5 --seed {seed} --graph-mode none

In [ ]:
!python compare_answers.py --examples 20
!python compare_answers.py --seeds 42 1 2

In [ ]:
!python relation_analysis.py --samples 3000
from IPython.display import Image, display
display(Image('results/relation_probes.png'))

In [ ]:
!python explain_reasoning.py --image images/Man_riding_bicycle.png --question "What is the man riding?"
!python explain_reasoning.py --image images/Man_riding_bicycle.png --question "What is the man holding?"
!python explain_reasoning.py --image images/giraffe.png --question "What animal is this?"
import glob
for f in sorted(glob.glob('results/reasoning/*.png')):
    print(f); display(Image(f))

## 9. Try the trained model on any image

In [ ]:
!python main.py --image images/Man_riding_bicycle.png --question "What is the man riding?"
!python whatif_qa.py --image images/Man_riding_bicycle.png --whatif "what if the bicycle were a motorcycle, what is the man riding?"

To use your own picture: upload it with the folder icon on the left (into `/content/scm/images/`),
then change the `--image` path above.

## 10. Copy the checkpoint to your laptop

It is already in Google Drive at `MyDrive/scm_vlm/checkpoints/scm_vlm_latest.pt`.
Download it from Drive and put it in the project's `checkpoints/` folder, or run the cell below.

In [ ]:
from google.colab import files
files.download('checkpoints/scm_vlm_latest.pt')